# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanhGiauTen/flyrankAI/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook maps the chosen content-refresh lane onto an explicit ML decision contract.

## 1. My lane as an ML task (type)

The product task is **ranking/scoring**: assign each eligible page an opportunity score and order pages for a fixed-capacity editorial review queue. A binary classifier can estimate a decline-risk proxy, but its probability is an intermediate signal rather than the final product. The delivered output remains a ranked list because the decision is *which pages should be reviewed first?*

In [1]:
from pathlib import Path
import pandas as pd

DATA_URL = 'https://raw.githubusercontent.com/KhanhGiauTen/flyrankAI/main/data/raw/content_refresh_anonymized.csv'
local_candidates = [Path('data/raw/content_refresh_anonymized.csv'), Path('../../data/raw/content_refresh_anonymized.csv')]
data_source = next((path for path in local_candidates if path.exists()), DATA_URL)
df = pd.read_csv(data_source)
task_contract = {'product_output': 'ranked review queue', 'model_support': 'binary proxy score', 'review_capacity_k': 50}
task_contract


{'product_output': 'ranked review queue',
 'model_support': 'binary proxy score',
 'review_capacity_k': 50}

## 2. Target or proxy

For the starter exercise, the proxy target is `is_declining_proxy = 1` when observed impressions fell by more than 20% from the previous 30-day window to the latest 30-day window (`trend_direction == 'down'`). It comes from measured search-performance windows, but it is still a **defined current-window proxy**, not a future outcome and not evidence that a refresh will help. `trend_direction`, `trend_pct`, and the two 30-day impression inputs must be excluded from model features because they directly define the answer. The capstone should replace this with a future-window outcome built strictly after the feature window.

In [2]:
modeling = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id').copy()
modeling['is_declining_proxy'] = modeling['trend_direction'].eq('down').astype('int8')
leakage_columns = ['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d']
print(modeling['is_declining_proxy'].value_counts().rename(index={0: 'not_down', 1: 'down'}))
print(f'Proxy base rate: {modeling.is_declining_proxy.mean():.1%}')
print('Excluded direct-answer fields:', ', '.join(leakage_columns))


is_declining_proxy
down        16262
not_down    13738
Name: count, dtype: int64
Proxy base rate: 54.2%
Excluded direct-answer fields: trend_direction, trend_pct, impressions_last_30d, impressions_prev_30d


## 3. Success metric

The primary metric is **Precision@50 on held-out clients** because an editor can review roughly 50 pages per batch and false positives consume that capacity. Before training, success is defined as beating the transparent rule baseline by at least **10 percentage points at K=50** on the same client-holdout split. I will also report the proxy base rate, average precision, and the number of eligible pages so that a strong top-50 result cannot hide weak coverage or an easy class balance.

In [3]:
metric_contract = pd.Series({
    'primary_metric': 'Precision@50',
    'validation': 'client-group holdout',
    'minimum_improvement_over_rule_pp': 10,
    'proxy_base_rate_pct': round(modeling['is_declining_proxy'].mean() * 100, 1),
    'review_capacity_k': 50,
})
metric_contract.to_frame('precommitted_value')


,precommitted_value
primary_metric,Precision@50
validation,client-group holdout
minimum_improvement_over_rule_pp,10
proxy_base_rate_pct,54.2
review_capacity_k,50


## 4. The unit of analysis, as a real dataframe

One row represents **one pseudonymized content item (page) in a trailing 90-day snapshot**. The identifiers are grouping keys only. The example below shows the review grain and a small set of signals known at scoring time; percentages such as `ctr` are stored on a 0–100 scale, so `0.76` means 0.76%.

In [4]:
unit_columns = [
    'content_id', 'client_id', 'content_type', 'impressions_90d', 'clicks_90d',
    'sessions_90d', 'ctr', 'avg_position', 'content_age_days',
    'days_since_last_update', 'is_declining_proxy'
]
unit = modeling[unit_columns].copy()
assert unit['content_id'].is_unique
print(f'Grain check: {len(unit):,} rows and {unit.content_id.nunique():,} unique content items.')
unit.head(8)


Grain check: 30,000 rows and 30,000 unique content items.


,content_id,client_id,content_type,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,content_age_days,days_since_last_update,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,17,0.76,10.6,187,20,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,9,0.05,20.3,445,25,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,11,0.09,36.5,141,20,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,78,0.49,6.2,463,22,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,145,0.13,44.0,263,14,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,5,0.03,8.5,147,20,1
6,content_9a34b442b552,client_8722616204,keyword article,20,0,1,0.00,7.0,90,20,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,28,0.06,21.2,445,22,0


## 5. Why ML beats a fixed rule here

A fixed rule is the required baseline, but it cannot adapt gracefully to interacting signals: the meaning of CTR changes with average position and impression volume; traffic distributions are heavy-tailed; missing keyword and word-count fields vary by content type; and client portfolios have different scales. A small, interpretable model can combine these relationships and produce a smoother ordering. ML earns its place only if it improves held-out top-K precision; otherwise the simpler rule should remain the decision tool.

In [5]:
heterogeneity = (
    modeling.groupby('content_type', dropna=False)
    .agg(
        rows=('content_id', 'size'),
        decline_proxy_rate=('is_declining_proxy', 'mean'),
        median_impressions=('impressions_90d', 'median'),
        keyword_missing_rate=('search_volume', lambda s: s.isna().mean()),
    )
)
heterogeneity['decline_proxy_rate'] = heterogeneity['decline_proxy_rate'].map(lambda x: f'{x:.1%}')
heterogeneity['keyword_missing_rate'] = heterogeneity['keyword_missing_rate'].map(lambda x: f'{x:.1%}')
heterogeneity


,rows,decline_proxy_rate,median_impressions,keyword_missing_rate
content_type,,,,
comparison article,697,57.2%,107.0,0.0%
feedly article,2096,28.7%,4.0,100.0%
keyword article,27207,56.1%,955.0,1.4%


## Self-check

- [x] Named the ML task type, proxy target, and decision metric
- [x] Showed the unit of analysis as a real dataframe
- [x] Explained why ML must beat a fixed-rule baseline to be justified
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries appear in the output
- [x] Claims are limited to observed, directional decision support
- [x] Saved under `work/notebooks/` for the repository submission